# Experimento A: Establecimiento de la Línea Base (Baseline Puro)

## Objetivo
Medir el rendimiento **"natural"** de los algoritmos sin ninguna ayuda para el desbalance.  
Esto sirve de **punto de referencia (suelo)** para cuantificar la mejora real de las técnicas posteriores.

## Modelos
- **Regresión Logística**: Modelo lineal simple de referencia
- **Random Forest (RF)**: Ensamble robusto estándar
- **XGBoost**: Estado del arte en boosting

## Configuración
- Parámetros por defecto de sklearn y xgboost
- **Sin** `class_weight`, **sin** SMOTE
- Preprocesamiento: `StandardScaler` en features numéricas
- División temporal estricta (protocolo del libro)

## Hipótesis
Se espera una Accuracy altísima (~99%) pero un Recall de fraude muy bajo (<20-30%),  
demostrando la inutilidad de la métrica Accuracy en problemas desbalanceados.

## Métrica Clave
- AUPRC (Area Under Precision-Recall Curve)
- Card Precision@100

---
## 1. Imports y Configuración

In [ ]:
import os
import sys
import datetime
import warnings
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn import metrics
import xgboost as xgb

# Configuración del proyecto
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from experiments.config import (
    SEED, INPUT_FEATURES, OUTPUT_FEATURE,
    BASELINE_PARAMS, RESULTS_DIR, FIGURES_DIR,
    TOP_K_LIST, COLORS,
    START_DATE_TRAINING, DELTA_TRAIN, DELTA_DELAY, DELTA_TEST,
)
from experiments.data_utils import (
    load_transformed_data, get_train_test_set,
    print_dataset_summary, card_precision_top_k,
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid', {'axes.facecolor': '0.9'})

print(f"Semilla de reproducibilidad: {SEED}")
print(f"Features de entrada: {len(INPUT_FEATURES)}")
print(f"Variable objetivo: {OUTPUT_FEATURE}")

---
## 2. Carga de Datos

In [ ]:
# Cargar dataset transformado (con feature engineering del Chapter 3)
transactions_df = load_transformed_data()
print(f"Dataset cargado: {len(transactions_df):,} transacciones")
print(f"Periodo: {transactions_df.TX_DATETIME.min()} → {transactions_df.TX_DATETIME.max()}")
print(f"\nColumnas disponibles ({len(transactions_df.columns)}):")
print(list(transactions_df.columns))

In [ ]:
# Crear conjuntos train/test con división temporal estricta
train_df, test_df = get_train_test_set(
    transactions_df,
    start_date_training=START_DATE_TRAINING,
    delta_train=DELTA_TRAIN,
    delta_delay=DELTA_DELAY,
    delta_test=DELTA_TEST,
)

print_dataset_summary(train_df, test_df, "Experimento A - Baseline Puro")

---
## 3. Entrenamiento de Modelos Baseline

Todos los modelos usan parámetros por defecto.  
**No se aplica ninguna técnica para manejar el desbalance.**

In [ ]:
def train_and_evaluate(classifier, name, train_df, test_df,
                        input_features, output_feature, top_k_list=None):
    """
    Entrena un modelo con pipeline de escalado y evalúa sobre test.
    Calcula métricas estándar + Card Precision@k (protocolo del libro).

    Args:
        classifier: Clasificador sklearn/xgboost
        name: Nombre descriptivo del modelo
        train_df: DataFrame de entrenamiento (con CUSTOMER_ID, TX_TIME_DAYS)
        test_df: DataFrame de test (con CUSTOMER_ID, TX_TIME_DAYS)
        input_features: Lista de features de entrada
        output_feature: Nombre de la variable objetivo
        top_k_list: Lista de k para Card Precision@k (por defecto [100])

    Returns:
        dict con predicciones, pipeline y todas las métricas
    """
    if top_k_list is None:
        top_k_list = TOP_K_LIST

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', classifier),
    ])

    pipeline.fit(train_df[input_features], train_df[output_feature])

    y_pred_proba_test = pipeline.predict_proba(test_df[input_features])[:, 1]
    y_pred_proba_train = pipeline.predict_proba(train_df[input_features])[:, 1]

    # Métricas estándar
    auc_roc = metrics.roc_auc_score(test_df[output_feature], y_pred_proba_test)
    avg_precision = metrics.average_precision_score(test_df[output_feature], y_pred_proba_test)

    y_pred_class = (y_pred_proba_test >= 0.5).astype(int)
    accuracy = metrics.accuracy_score(test_df[output_feature], y_pred_class)
    recall = metrics.recall_score(test_df[output_feature], y_pred_class)
    precision = metrics.precision_score(test_df[output_feature], y_pred_class, zero_division=0)
    f1 = metrics.f1_score(test_df[output_feature], y_pred_class)

    # Card Precision@k (protocolo del libro, Chapter 4)
    predictions_df = test_df.copy()
    predictions_df['predictions'] = y_pred_proba_test

    cp_at_k = {}
    for k in top_k_list:
        _, _, mean_cp = card_precision_top_k(predictions_df, k)
        cp_at_k[k] = mean_cp

    result = {
        'name': name,
        'pipeline': pipeline,
        'y_pred_proba_test': y_pred_proba_test,
        'y_pred_proba_train': y_pred_proba_train,
        'auc_roc': auc_roc,
        'avg_precision': avg_precision,
        'accuracy': accuracy,
        'recall': recall,
        'precision': precision,
        'f1': f1,
        'card_precision_at_k': cp_at_k,
    }

    print(f"\n  {name}:")
    print(f"    AUC ROC:           {auc_roc:.4f}")
    print(f"    AUPRC (Avg Prec):  {avg_precision:.4f}")
    for k, cp in cp_at_k.items():
        print(f"    Card Prec@{k}:     {cp:.4f}")
    print(f"    Accuracy:          {accuracy:.4f}  (↑ engañosamente alta)")
    print(f"    Recall (fraude):   {recall:.4f}  (↓ capacidad real de detección)")
    print(f"    Precision:         {precision:.4f}")
    print(f"    F1-Score:          {f1:.4f}")

    return result

In [ ]:
# Definir clasificadores con parámetros por defecto (sin ajuste de desbalance)
classifiers = {
    "Logistic Regression": LogisticRegression(**BASELINE_PARAMS["Logistic Regression"]),
    "Random Forest": RandomForestClassifier(**BASELINE_PARAMS["Random Forest"]),
    "XGBoost": xgb.XGBClassifier(**BASELINE_PARAMS["XGBoost"]),
}

print("=" * 60)
print("  RESULTADOS DEL EXPERIMENTO A: BASELINE PURO")
print("=" * 60)

results_a = {}
for name, clf in classifiers.items():
    results_a[name] = train_and_evaluate(
        clf, name, train_df, test_df,
        INPUT_FEATURES, OUTPUT_FEATURE,
    )

---
## 4. Tabla Comparativa y Visualizaciones

In [ ]:
# Tabla de resultados (con Card Precision@100)
results_table = pd.DataFrame({
    'Modelo': [r['name'] for r in results_a.values()],
    'AUC ROC': [r['auc_roc'] for r in results_a.values()],
    'AUPRC': [r['avg_precision'] for r in results_a.values()],
    'CP@100': [r['card_precision_at_k'][100] for r in results_a.values()],
    'Accuracy': [r['accuracy'] for r in results_a.values()],
    'Recall (Fraude)': [r['recall'] for r in results_a.values()],
    'Precision': [r['precision'] for r in results_a.values()],
    'F1-Score': [r['f1'] for r in results_a.values()],
}).set_index('Modelo').round(4)

print("\nTabla comparativa - Experimento A (Baseline Puro):")
print("=" * 80)
display(results_table)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Curva Precision-Recall
ax = axes[0]
for name, res in results_a.items():
    prec, rec, _ = metrics.precision_recall_curve(
        test_df[OUTPUT_FEATURE], res['y_pred_proba_test'])
    ax.plot(rec, prec, label=f"{name} (AP={res['avg_precision']:.3f})")
baseline_fraud_rate = test_df[OUTPUT_FEATURE].mean()
ax.axhline(y=baseline_fraud_rate, color='r', linestyle='--',
           label=f'Aleatorio (AP={baseline_fraud_rate:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Curva Precision-Recall\n(Experimento A)', fontsize=14)
ax.legend(fontsize=9)
ax.set_xlim([0, 1.01]); ax.set_ylim([0, 1.01])

# Curva ROC
ax = axes[1]
for name, res in results_a.items():
    fpr, tpr, _ = metrics.roc_curve(
        test_df[OUTPUT_FEATURE], res['y_pred_proba_test'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc_roc']:.3f})")
ax.plot([0, 1], [0, 1], 'r--', label='Aleatorio (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Curva ROC\n(Experimento A)', fontsize=14)
ax.legend(fontsize=9)

# Accuracy vs Recall (paradoja del desbalance)
ax = axes[2]
model_names = list(results_a.keys())
accuracies = [results_a[n]['accuracy'] for n in model_names]
recalls = [results_a[n]['recall'] for n in model_names]
x_pos = np.arange(len(model_names))
width = 0.35
bars1 = ax.bar(x_pos - width/2, accuracies, width, label='Accuracy', color=COLORS['baseline'])
bars2 = ax.bar(x_pos + width/2, recalls, width, label='Recall (Fraude)', color=COLORS['incorrect_pipeline'])
ax.set_ylabel('Valor', fontsize=12)
ax.set_title('Accuracy vs Recall\n(Paradoja del desbalance)', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.legend(); ax.set_ylim([0, 1.1])
for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()), ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_a_baseline_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nFigura guardada en: {FIGURES_DIR / 'experiment_a_baseline_results.png'}")

---
## 5. Guardar Resultados

In [ ]:
# Guardar tabla de resultados
results_table.to_csv(RESULTS_DIR / 'experiment_a_results.csv')

# Guardar predicciones y métricas para comparar con Exp B y C
results_to_save = {
    name: {
        'auc_roc': res['auc_roc'],
        'avg_precision': res['avg_precision'],
        'card_precision_at_k': res['card_precision_at_k'],
        'accuracy': res['accuracy'],
        'recall': res['recall'],
        'precision': res['precision'],
        'f1': res['f1'],
        'y_pred_proba_test': res['y_pred_proba_test'],
    }
    for name, res in results_a.items()
}

with open(RESULTS_DIR / 'experiment_a_predictions.pkl', 'wb') as f:
    pickle.dump(results_to_save, f)

print("✓ Resultados del Experimento A guardados exitosamente")
print(f"  - CSV: {RESULTS_DIR / 'experiment_a_results.csv'}")
print(f"  - PKL: {RESULTS_DIR / 'experiment_a_predictions.pkl'}")

---
## 6. Conclusiones del Experimento A

**Validación de la hipótesis:**

- La **Accuracy** es engañosamente alta (~99%) porque el modelo simplemente predice "no fraude" la mayoría del tiempo.
- El **Recall** de la clase fraude es muy bajo, lo que significa que el modelo falla en detectar los casos que realmente importan.
- La **AUPRC** es la métrica que mejor refleja el rendimiento real en este contexto desbalanceado.

Estos resultados establecen el **suelo de rendimiento** contra el cual se compararán los Experimentos B, C y D.